## Convert MATLAB Pixel Labels to COCO Format: One-Class Bubble Model

This notebook converts MATLAB-exported pixel label masks into COCO-format annotation files for Detectron2 Mask R-CNN training. This version is intended for datasets where all bubble instances are labeled as a single class: `bubble`.

The script pairs each training image with its matching MATLAB pixel label mask, separates disconnected labeled regions into individual bubble instances, extracts polygon segmentations and bounding boxes, and writes the annotations to `train.json` and `val.json`.

Expected indexed mask format:

```text
0 = background
1 = bubble

In [1]:
# ============================================================
# Convert MATLAB exported pixel label masks to COCO JSON
# for Detectron2 Mask R-CNN training.
#
# One-class version.
#
# Output:
#   train.json
#   val.json
#
# Classes:
#   1 = bubble
#
# No test set is created.
# ============================================================

import json
import random
from pathlib import Path

import numpy as np
from PIL import Image
from skimage import measure


# ------------------------------------------------------------
# USER SETTINGS - CHANGE THESE PATHS
# ------------------------------------------------------------

IMAGE_DIR = Path(r"C:\path\to\PNG Training Images")

MASK_DIR = Path(r"C:\path\to\PixelLabelData\pixelLabelData")

OUTPUT_DIR = Path(r"C:\path\to\coco_annotations_one_class")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# SPLIT SETTINGS
# ------------------------------------------------------------

TRAIN_RATIO = 0.85
RANDOM_SEED = 1
MIN_AREA = 20


# ------------------------------------------------------------
# CLASS DEFINITIONS
# ------------------------------------------------------------

CATEGORIES = [
    {"id": 1, "name": "bubble", "supercategory": "bubble"},
]


# ------------------------------------------------------------
# MASK LABEL SETTINGS
# ------------------------------------------------------------
# Most MATLAB pixel label exports are indexed masks:
#   0 = background
#   1 = bubble
#
# If your MATLAB label is named "Bubble", the exported indexed
# mask usually has:
#   mask value 0 -> background
#   mask value 1 -> bubble
# ------------------------------------------------------------

MASK_MODE = "indexed"

INDEXED_LABEL_MAP = {
    1: 1,  # MATLAB mask value 1 -> COCO category 1 bubble
}

# Only used if your masks are RGB instead of indexed.
# Change this color if MATLAB exported a different RGB label color.
RGB_LABEL_MAP = {
    (0, 255, 0): 1,
}


# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------

def find_matching_mask(image_path, mask_dir):
    """
    Finds a MATLAB exported pixel label mask for an image.

    Handles masks named like:
        Label_100_bubble_0100.png

    for images named like:
        bubble_0100.png
    """
    base = image_path.stem

    possible_extensions = [".png", ".tif", ".tiff"]

    # First try exact filename match.
    for ext in possible_extensions:
        candidate = mask_dir / f"{base}{ext}"
        if candidate.exists():
            return candidate

    # Then try MATLAB's Label_* prefix pattern.
    for ext in possible_extensions:
        matches = list(mask_dir.glob(f"Label_*_{base}{ext}"))

        if len(matches) == 1:
            return matches[0]

        if len(matches) > 1:
            raise RuntimeError(
                f"Multiple masks found for {image_path.name}: {matches}"
            )

    return None


def load_mask(mask_path):
    """
    Loads mask image.
    """
    mask = np.array(Image.open(mask_path))

    if mask.ndim == 3 and mask.shape[2] == 4:
        mask = mask[:, :, :3]

    return mask


def load_binary_masks_by_class(mask_path):
    """
    Converts one MATLAB pixel label mask into binary masks for each class.

    Returns:
        dict {category_id: binary_mask}
    """
    mask = load_mask(mask_path)
    class_masks = {}

    if MASK_MODE == "indexed":
        if mask.ndim == 3:
            raise ValueError(
                f"{mask_path.name} opened as RGB, but MASK_MODE is set to 'indexed'. "
                "Change MASK_MODE to 'rgb' and update RGB_LABEL_MAP."
            )

        for matlab_value, category_id in INDEXED_LABEL_MAP.items():
            class_masks[category_id] = (mask == matlab_value).astype(np.uint8)

    elif MASK_MODE == "rgb":
        if mask.ndim != 3:
            raise ValueError(
                f"{mask_path.name} opened as grayscale/indexed, but MASK_MODE is set to 'rgb'. "
                "Change MASK_MODE to 'indexed'."
            )

        mask_rgb = mask[:, :, :3]

        for rgb_color, category_id in RGB_LABEL_MAP.items():
            color = np.array(rgb_color, dtype=np.uint8)
            class_masks[category_id] = np.all(mask_rgb == color, axis=-1).astype(np.uint8)

    else:
        raise ValueError("MASK_MODE must be 'indexed' or 'rgb'.")

    return class_masks


def get_image_size(image_path):
    """
    Returns width, height.
    """
    with Image.open(image_path) as img:
        return img.size


def get_bbox_from_binary_mask(binary_mask):
    """
    Gets COCO bbox format:
        [x, y, width, height]
    """
    ys, xs = np.where(binary_mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min = float(xs.min())
    y_min = float(ys.min())
    x_max = float(xs.max())
    y_max = float(ys.max())

    width = x_max - x_min + 1.0
    height = y_max - y_min + 1.0

    return [x_min, y_min, width, height]


def instance_mask_to_polygons(instance_mask):
    """
    Converts one binary instance mask to COCO polygon segmentation.
    """
    polygons = []

    contours = measure.find_contours(instance_mask, 0.5)

    for contour in contours:
        if len(contour) < 3:
            continue

        # skimage contour format is [row, col].
        # COCO needs [x, y].
        contour = np.flip(contour, axis=1)

        # Simplify very long contours to keep JSON smaller.
        if len(contour) > 150:
            step = max(1, len(contour) // 150)
            contour = contour[::step]

        segmentation = contour.ravel().tolist()

        if len(segmentation) >= 6:
            polygons.append(segmentation)

    return polygons


def create_coco_json(pairs):
    """
    Creates COCO dictionary from image/mask pairs.
    """
    coco = {
        "images": [],
        "annotations": [],
        "categories": CATEGORIES,
    }

    image_id = 1
    annotation_id = 1

    for image_path, mask_path in pairs:
        width, height = get_image_size(image_path)

        coco["images"].append({
            "id": image_id,
            "file_name": image_path.name,
            "width": width,
            "height": height,
        })

        class_masks = load_binary_masks_by_class(mask_path)

        for category_id, binary_mask in class_masks.items():
            if binary_mask.sum() == 0:
                continue

            # Split disconnected blobs into individual bubble instances.
            labeled_mask = measure.label(binary_mask, connectivity=2)

            for region in measure.regionprops(labeled_mask):
                if region.area < MIN_AREA:
                    continue

                instance_mask = (labeled_mask == region.label).astype(np.uint8)

                bbox = get_bbox_from_binary_mask(instance_mask)

                if bbox is None:
                    continue

                polygons = instance_mask_to_polygons(instance_mask)

                if len(polygons) == 0:
                    continue

                area = int(instance_mask.sum())

                coco["annotations"].append({
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": int(category_id),
                    "segmentation": polygons,
                    "area": area,
                    "bbox": bbox,
                    "iscrowd": 0,
                })

                annotation_id += 1

        image_id += 1

    return coco


def print_class_counts(coco, dataset_name):
    """
    Prints bubble annotation counts.
    """
    count = 0

    for ann in coco["annotations"]:
        if ann["category_id"] == 1:
            count += 1

    print(f"\n{dataset_name} class counts:")
    print(f"  bubble: {count}")


def inspect_first_mask(mask_dir):
    """
    Prints unique mask values so you can confirm the label map.
    """
    mask_files = sorted(
        list(mask_dir.glob("*.png")) +
        list(mask_dir.glob("*.tif")) +
        list(mask_dir.glob("*.tiff"))
    )

    if len(mask_files) == 0:
        print(f"No masks found in {mask_dir}")
        return

    sample_path = mask_files[0]
    mask = load_mask(sample_path)

    print("\nSample mask inspection:")
    print(f"  File: {sample_path.name}")
    print(f"  Shape: {mask.shape}")
    print(f"  Dtype: {mask.dtype}")

    if mask.ndim == 2:
        print("  Unique values:")
        print(f"  {np.unique(mask)}")
    else:
        pixels = mask.reshape(-1, mask.shape[-1])
        unique_colors = np.unique(pixels, axis=0)
        print("  Unique colors, first 20:")
        print(unique_colors[:20])
        print(f"  Number of unique colors: {len(unique_colors)}")


# ------------------------------------------------------------
# FIND IMAGE AND MASK FILES
# ------------------------------------------------------------

print("Inspecting masks...")
inspect_first_mask(MASK_DIR)

image_files = sorted(
    list(IMAGE_DIR.glob("*.png")) +
    list(IMAGE_DIR.glob("*.jpg")) +
    list(IMAGE_DIR.glob("*.jpeg")) +
    list(IMAGE_DIR.glob("*.tif")) +
    list(IMAGE_DIR.glob("*.tiff"))
)

if len(image_files) == 0:
    raise FileNotFoundError(f"No image files found in: {IMAGE_DIR}")

print(f"\nFound {len(image_files)} image files.")

paired_files = []

for image_path in image_files:
    mask_path = find_matching_mask(image_path, MASK_DIR)

    if mask_path is None:
        print(f"Warning: no matching mask found for {image_path.name}")
        continue

    paired_files.append((image_path, mask_path))

print(f"Found {len(paired_files)} image/mask pairs.")

if len(paired_files) == 0:
    raise RuntimeError(
        "No image/mask pairs found. "
        "Make sure image files and mask files have matching base names."
    )


# ------------------------------------------------------------
# TRAIN / VAL SPLIT
# ------------------------------------------------------------

random.seed(RANDOM_SEED)
random.shuffle(paired_files)

num_train = int(round(TRAIN_RATIO * len(paired_files)))

train_pairs = paired_files[:num_train]
val_pairs = paired_files[num_train:]

print("\nSplit:")
print(f"  Train images: {len(train_pairs)}")
print(f"  Val images:   {len(val_pairs)}")


# ------------------------------------------------------------
# CREATE COCO JSON FILES
# ------------------------------------------------------------

train_coco = create_coco_json(train_pairs)
val_coco = create_coco_json(val_pairs)

train_json_path = OUTPUT_DIR / "train.json"
val_json_path = OUTPUT_DIR / "val.json"

with open(train_json_path, "w") as f:
    json.dump(train_coco, f, indent=2)

with open(val_json_path, "w") as f:
    json.dump(val_coco, f, indent=2)

print("\nSaved COCO JSON files:")
print(f"  {train_json_path}")
print(f"  {val_json_path}")

print("\nAnnotation totals:")
print(f"  Train images:      {len(train_coco['images'])}")
print(f"  Train annotations: {len(train_coco['annotations'])}")
print(f"  Val images:        {len(val_coco['images'])}")
print(f"  Val annotations:   {len(val_coco['annotations'])}")

print_class_counts(train_coco, "Train")
print_class_counts(val_coco, "Validation")


# ------------------------------------------------------------
# WARNINGS
# ------------------------------------------------------------

if len(train_coco["annotations"]) == 0:
    print("\nWARNING: train.json has zero annotations.")

if len(val_coco["annotations"]) == 0:
    print("\nWARNING: val.json has zero annotations.")

train_count = sum(
    1 for ann in train_coco["annotations"]
    if ann["category_id"] == 1
)

val_count = sum(
    1 for ann in val_coco["annotations"]
    if ann["category_id"] == 1
)

if train_count == 0:
    print("\nWARNING: No bubble annotations found in the training set.")

if val_count == 0:
    print("\nWARNING: No bubble annotations found in the validation set.")

Inspecting masks...
No masks found in C:\path\to\PixelLabelData\pixelLabelData


FileNotFoundError: No image files found in: C:\path\to\PNG Training Images

### Convert MATLAB Pixel Labels to COCO Format: Attached/Detached Bubble Model

This notebook converts MATLAB-exported pixel label masks into COCO-format annotation files for Detectron2 Mask R-CNN training. This version is intended for datasets where bubbles are separated into two classes: `attached_bubble` and `detached_bubble`.

The script pairs each training image with its matching MATLAB pixel label mask, separates disconnected labeled regions into individual bubble instances, extracts polygon segmentations and bounding boxes, and writes the annotations to `train.json` and `val.json`.

Expected indexed mask format:

0 = background
1 = attached_bubble
2 = detached_bubble

In [ ]:
# ============================================================
# Convert MATLAB exported pixel label masks to COCO JSON
# for Detectron2 Mask R-CNN training.
#
# Output:
#   train.json
#   val.json
#
# Classes:
#   1 = attached_bubble
#   2 = detached_bubble
#
# No test set is created.
# ============================================================

import os
import json
import random
from pathlib import Path

import numpy as np
from PIL import Image
from skimage import measure


# ------------------------------------------------------------
# USER SETTINGS - CHANGE THESE PATHS
# ------------------------------------------------------------

IMAGE_DIR = Path(r"C:\path\to\PNG Training Images")

MASK_DIR = Path(r"C:\path\to\PixelLabelData\pixelLabelData")

OUTPUT_DIR = Path(r"C:\path\to\coco_annotations")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# SPLIT SETTINGS
# ------------------------------------------------------------

TRAIN_RATIO = 0.85
RANDOM_SEED = 1
MIN_AREA = 20


# ------------------------------------------------------------
# CLASS DEFINITIONS
# ------------------------------------------------------------

CATEGORIES = [
    {"id": 1, "name": "attached_bubble", "supercategory": "bubble"},
    {"id": 2, "name": "detached_bubble", "supercategory": "bubble"},
]


# ------------------------------------------------------------
# MASK LABEL SETTINGS
# ------------------------------------------------------------
# Most MATLAB pixel label exports are indexed masks:
#   0 = background
#   1 = first label
#   2 = second label
#
# Based on your MATLAB label list:
#   first label  = attached bubble
#   second label = detached_bubble
#
# So:
#   mask value 1 -> attached_bubble
#   mask value 2 -> detached_bubble
# ------------------------------------------------------------

MASK_MODE = "indexed"

INDEXED_LABEL_MAP = {
    1: 1,  # MATLAB mask value 1 -> COCO category 1 attached_bubble
    2: 2,  # MATLAB mask value 2 -> COCO category 2 detached_bubble
}

# Only used if your masks are RGB instead of indexed.
# You probably do not need this.
RGB_LABEL_MAP = {
    (0, 255, 0): 1,
    (255, 128, 0): 2,
}


# ------------------------------------------------------------
# HELPER FUNCTIONS
# ------------------------------------------------------------

def find_matching_mask(image_path, mask_dir):
    """
    Finds a MATLAB exported pixel label mask for an image.

    Handles masks named like:
        Label_100_attached_detached_0100.png

    for images named like:
        attached_detached_0100.png
    """
    base = image_path.stem

    possible_extensions = [".png", ".tif", ".tiff"]

    # First try exact filename match
    for ext in possible_extensions:
        candidate = mask_dir / f"{base}{ext}"
        if candidate.exists():
            return candidate

    # Then try MATLAB's Label_* prefix pattern
    for ext in possible_extensions:
        matches = list(mask_dir.glob(f"Label_*_{base}{ext}"))
        if len(matches) == 1:
            return matches[0]
        elif len(matches) > 1:
            raise RuntimeError(f"Multiple masks found for {image_path.name}: {matches}")

    return None


def load_mask(mask_path):
    """
    Loads mask image.
    """
    mask = np.array(Image.open(mask_path))

    if mask.ndim == 3 and mask.shape[2] == 4:
        mask = mask[:, :, :3]

    return mask


def load_binary_masks_by_class(mask_path):
    """
    Converts one MATLAB pixel label mask into binary masks for each class.

    Returns:
        dict {category_id: binary_mask}
    """
    mask = load_mask(mask_path)
    class_masks = {}

    if MASK_MODE == "indexed":
        if mask.ndim == 3:
            raise ValueError(
                f"{mask_path.name} opened as RGB, but MASK_MODE is set to 'indexed'. "
                "Change MASK_MODE to 'rgb' and update RGB_LABEL_MAP."
            )

        for matlab_value, category_id in INDEXED_LABEL_MAP.items():
            class_masks[category_id] = (mask == matlab_value).astype(np.uint8)

    elif MASK_MODE == "rgb":
        if mask.ndim != 3:
            raise ValueError(
                f"{mask_path.name} opened as grayscale/indexed, but MASK_MODE is set to 'rgb'. "
                "Change MASK_MODE to 'indexed'."
            )

        mask_rgb = mask[:, :, :3]

        for rgb_color, category_id in RGB_LABEL_MAP.items():
            color = np.array(rgb_color, dtype=np.uint8)
            class_masks[category_id] = np.all(mask_rgb == color, axis=-1).astype(np.uint8)

    else:
        raise ValueError("MASK_MODE must be 'indexed' or 'rgb'.")

    return class_masks


def get_image_size(image_path):
    """
    Returns width, height.
    """
    with Image.open(image_path) as img:
        return img.size


def get_bbox_from_binary_mask(binary_mask):
    """
    Gets COCO bbox format:
        [x, y, width, height]
    """
    ys, xs = np.where(binary_mask > 0)

    if len(xs) == 0 or len(ys) == 0:
        return None

    x_min = float(xs.min())
    y_min = float(ys.min())
    x_max = float(xs.max())
    y_max = float(ys.max())

    width = x_max - x_min + 1.0
    height = y_max - y_min + 1.0

    return [x_min, y_min, width, height]


def instance_mask_to_polygons(instance_mask):
    """
    Converts one binary instance mask to COCO polygon segmentation.
    """
    polygons = []

    contours = measure.find_contours(instance_mask, 0.5)

    for contour in contours:
        if len(contour) < 3:
            continue

        # skimage contour format is [row, col].
        # COCO needs [x, y].
        contour = np.flip(contour, axis=1)

        # Simplify very long contours to keep JSON smaller.
        if len(contour) > 150:
            step = max(1, len(contour) // 150)
            contour = contour[::step]

        segmentation = contour.ravel().tolist()

        if len(segmentation) >= 6:
            polygons.append(segmentation)

    return polygons


def create_coco_json(pairs):
    """
    Creates COCO dictionary from image/mask pairs.
    """
    coco = {
        "images": [],
        "annotations": [],
        "categories": CATEGORIES,
    }

    image_id = 1
    annotation_id = 1

    for image_path, mask_path in pairs:
        width, height = get_image_size(image_path)

        coco["images"].append({
            "id": image_id,
            "file_name": image_path.name,
            "width": width,
            "height": height,
        })

        class_masks = load_binary_masks_by_class(mask_path)

        for category_id, binary_mask in class_masks.items():

            if binary_mask.sum() == 0:
                continue

            # Split disconnected blobs into individual bubble instances.
            labeled_mask = measure.label(binary_mask, connectivity=2)

            for region in measure.regionprops(labeled_mask):

                if region.area < MIN_AREA:
                    continue

                instance_mask = (labeled_mask == region.label).astype(np.uint8)

                bbox = get_bbox_from_binary_mask(instance_mask)

                if bbox is None:
                    continue

                polygons = instance_mask_to_polygons(instance_mask)

                if len(polygons) == 0:
                    continue

                area = int(instance_mask.sum())

                coco["annotations"].append({
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": int(category_id),
                    "segmentation": polygons,
                    "area": area,
                    "bbox": bbox,
                    "iscrowd": 0,
                })

                annotation_id += 1

        image_id += 1

    return coco


def print_class_counts(coco, dataset_name):
    """
    Prints attached/detached annotation counts.
    """
    counts = {1: 0, 2: 0}

    for ann in coco["annotations"]:
        counts[ann["category_id"]] += 1

    print(f"\n{dataset_name} class counts:")
    print(f"  attached_bubble: {counts[1]}")
    print(f"  detached_bubble: {counts[2]}")


def inspect_first_mask(mask_dir):
    """
    Prints unique mask values so you can confirm the label map.
    """
    mask_files = sorted(
        list(mask_dir.glob("*.png")) +
        list(mask_dir.glob("*.tif")) +
        list(mask_dir.glob("*.tiff"))
    )

    if len(mask_files) == 0:
        print(f"No masks found in {mask_dir}")
        return

    sample_path = mask_files[0]
    mask = load_mask(sample_path)

    print("\nSample mask inspection:")
    print(f"  File: {sample_path.name}")
    print(f"  Shape: {mask.shape}")
    print(f"  Dtype: {mask.dtype}")

    if mask.ndim == 2:
        print("  Unique values:")
        print(f"  {np.unique(mask)}")
    else:
        pixels = mask.reshape(-1, mask.shape[-1])
        unique_colors = np.unique(pixels, axis=0)
        print("  Unique colors, first 20:")
        print(unique_colors[:20])
        print(f"  Number of unique colors: {len(unique_colors)}")


# ------------------------------------------------------------
# FIND IMAGE AND MASK FILES
# ------------------------------------------------------------

print("Inspecting masks...")
inspect_first_mask(MASK_DIR)

image_files = sorted(
    list(IMAGE_DIR.glob("*.png")) +
    list(IMAGE_DIR.glob("*.jpg")) +
    list(IMAGE_DIR.glob("*.jpeg")) +
    list(IMAGE_DIR.glob("*.tif")) +
    list(IMAGE_DIR.glob("*.tiff"))
)

if len(image_files) == 0:
    raise FileNotFoundError(f"No image files found in: {IMAGE_DIR}")

print(f"\nFound {len(image_files)} image files.")

paired_files = []

for image_path in image_files:
    mask_path = find_matching_mask(image_path, MASK_DIR)

    if mask_path is None:
        print(f"Warning: no matching mask found for {image_path.name}")
        continue

    paired_files.append((image_path, mask_path))

print(f"Found {len(paired_files)} image/mask pairs.")

if len(paired_files) == 0:
    raise RuntimeError(
        "No image/mask pairs found. "
        "Make sure image files and mask files have matching base names."
    )


# ------------------------------------------------------------
# TRAIN / VAL SPLIT
# ------------------------------------------------------------

random.seed(RANDOM_SEED)
random.shuffle(paired_files)

num_train = int(round(TRAIN_RATIO * len(paired_files)))

train_pairs = paired_files[:num_train]
val_pairs = paired_files[num_train:]

print("\nSplit:")
print(f"  Train images: {len(train_pairs)}")
print(f"  Val images:   {len(val_pairs)}")


# ------------------------------------------------------------
# CREATE COCO JSON FILES
# ------------------------------------------------------------

train_coco = create_coco_json(train_pairs)
val_coco = create_coco_json(val_pairs)

train_json_path = OUTPUT_DIR / "train.json"
val_json_path = OUTPUT_DIR / "val.json"

with open(train_json_path, "w") as f:
    json.dump(train_coco, f, indent=2)

with open(val_json_path, "w") as f:
    json.dump(val_coco, f, indent=2)

print("\nSaved COCO JSON files:")
print(f"  {train_json_path}")
print(f"  {val_json_path}")

print("\nAnnotation totals:")
print(f"  Train images:      {len(train_coco['images'])}")
print(f"  Train annotations: {len(train_coco['annotations'])}")
print(f"  Val images:        {len(val_coco['images'])}")
print(f"  Val annotations:   {len(val_coco['annotations'])}")

print_class_counts(train_coco, "Train")
print_class_counts(val_coco, "Validation")


# ------------------------------------------------------------
# WARNINGS
# ------------------------------------------------------------

if len(train_coco["annotations"]) == 0:
    print("\nWARNING: train.json has zero annotations.")

if len(val_coco["annotations"]) == 0:
    print("\nWARNING: val.json has zero annotations.")

train_counts = {1: 0, 2: 0}
for ann in train_coco["annotations"]:
    train_counts[ann["category_id"]] += 1

val_counts = {1: 0, 2: 0}
for ann in val_coco["annotations"]:
    val_counts[ann["category_id"]] += 1

if train_counts[1] == 0 or train_counts[2] == 0:
    print("\nWARNING: One class is missing from the training set.")

if val_counts[1] == 0 or val_counts[2] == 0:
    print("\nWARNING: One class is missing from the validation set.")